📅 **论文年份 (Year):2016 年**  
*Identity Mappings in Deep Residual Networks — He, Zhang, Ren, Sun*

# Paper 15: Identity Mappings in Deep Residual Networks(深度残差网络中的恒等映射)
## Kaiming He, Xiangyu Zhang, Shaoqing Ren, Jian Sun (2016)(何恺明、张祥雨、任少卿、孙剑(2016))

### Pre-activation ResNet(预激活 ResNet)

Improved residual blocks with better gradient flow. Key insight: move activation BEFORE convolution!

改进的残差块(residual block)具有更好的梯度流动。关键洞见:把激活(activation)移到卷积(convolution)之前!

## 📖 论文导读

**🎯 这篇文章想解决什么问题(目的):** 2015 年的原版 ResNet 已经证明"残差连接"能训练上百层的网络,但当层数继续加深(比如超过 1000 层)时,训练又开始变难、效果反而下降。作者想弄清楚:残差连接到底为什么有效?捷径(shortcut)上到底该不该放东西?怎样设计残差块,才能让信息像走高速公路一样,畅通无阻地从第一层传到最后一层?

**💡 主要贡献:** 这篇论文给出了一个非常干净的答案:捷径上什么都别放,保持"恒等映射"(identity mapping,即原样传递)最好。作者通过数学推导和大量对比实验证明,只要捷径是纯粹的直通道,正向的信号和反向的梯度都能无损地在任意两层之间传递。他们还提出了新的"预激活"(pre-activation)残差块:把批归一化(BN)和 ReLU 激活挪到卷积之前,而不是之后。

**🔧 方法:** 打个比方,原版残差块像是在高速公路的出口处设了一个收费站(相加之后还要过一次 ReLU),多少会拦截一部分信息;新设计把所有"关卡"都移进了辅路(残差分支内部),主路完全畅通。作者系统地测试了各种捷径变体——乘以常数、加门控、加 1×1 卷积、加 dropout——结果全都不如最简单的直通。凭借这个改动,他们成功训练了 1001 层的超深网络,在 CIFAR-10 上取得了当时最好的成绩。

**🌟 意义:** 这项工作把"残差学习"从一个巧妙的技巧升华为一条设计原则:让梯度有一条不受阻碍的直通路径。它是深度学习史上"把网络做深"这条主线的关键一环,而"先归一化、再变换、最后直连相加"的思想直接影响了后来的 Transformer(Pre-LN 结构)以及 GPT 等大模型的架构设计。读懂这篇论文,你就能理解为什么今天几乎所有大模型内部都是一层层"残差块"堆起来的。

## 🎯 核心结论 (Key Takeaways)

- **论文核心发现:捷径上什么都别放。** 作者从数学上证明,只有当 shortcut 是纯粹的恒等映射时,信号和梯度才能在任意两层之间无损传递;乘常数、加门控、加 1×1 卷积、加 dropout 等各种"改装"全都比不过最简单的直通。

- **一个小改动带来大提升:预激活 (pre-activation)。** 把 BN 和 ReLU 挪到卷积之前、并去掉相加之后的 ReLU,恒等路径就彻底畅通了。凭借这个改动,论文在 CIFAR-10 上成功训练了 **1001 层**的超深网络并取得当时最好成绩,而原版结构在这种深度下已经难以优化。

- **本 notebook 的梯度实验验证了这一点:** 模拟梯度反向穿过 20 层残差块时,原版块(加法后的 ReLU 会"杀死"部分梯度,每层残差路径只保留约 50%~100%)传到输入层的梯度明显弱于预激活块(保留约 70%~100%)——预激活让深层网络的梯度流更强、更容易训练。

- **"恒等映射考试"给出最直接的证据:** 把残差路径权重全部清零后,用 100 个随机输入测试,预激活块的输出几乎严格等于输入(误差接近 0),而原版块因为加法后的 ReLU 把负值截断,输出无法还原输入——它的"恒等路径"其实并不恒等。

- **四种激活位置变体的对照表显示:** 只有"完全预激活"(x → BN → ReLU → Conv → BN → ReLU → Conv → +x)的恒等路径完全干净(★★★★★),原版、BN 后置、ReLU 前置三种变体的直通路径都被 BN 或 ReLU 不同程度地堵住。

- **带走一句话:** 想把网络做深,就给梯度留一条不受任何阻碍的直通高速路——这条"先归一化、再变换、最后直连相加"的原则,直接演化成了 Transformer 的 Pre-LN 结构,是今天大模型都在用的设计基因。

## 🤯 反常识的发现 (Counterintuitive Findings)

- **常识认为:给捷径加点"聪明"的机制(门控、1×1 卷积)应该比傻乎乎的直通更强。** 但论文把 5 种改装方案全试了一遍,结果连最温和的"乘个 0.5"都会让 110 层网络的错误率明显上升,门控和 1×1 卷积甚至直接训练失败——捷径上加任何东西都是减分项,**保持通道完全干净、什么都不放才是最优解**。

- **常识认为:每个模块输出后接一个 ReLU 是天经地义的标配,放哪儿无所谓。** 但实验发现加法之后的那个 ReLU 恰恰是"堵路"的罪魁祸首:本 notebook 的"恒等映射考试"显示,把残差路径权重全部清零后,原版块的输出仍无法还原输入(负值被 ReLU 截断了),它引以为傲的"恒等路径"其实并不恒等;而预激活块的误差几乎为 0。

- **常识认为:让 1001 层的网络能训练,得靠什么全新的重大发明。** 但论文只是把 BN 和 ReLU 从卷积后面挪到前面(pre-activation)——一个连一行新公式都没有的"调换顺序"小改动——就让 1001 层网络顺利收敛并刷新 CIFAR-10 纪录。notebook 里 20 层的梯度流实验也验证了:预激活块传回输入层的梯度明显更强、更稳。

- **常识认为:加正则(比如 dropout)总归对泛化有好处,加在哪儿都行。** 但论文发现把 dropout 放到捷径上反而让网络更难训练、效果更差——因为它在"高速公路"上随机设卡,梯度的无损直通被破坏了。正则要加也只能加在残差路径里,直通路必须神圣不可侵犯。


#### 💻 代码解读

**做什么:** 导入本笔记本需要的工具库,并固定随机种子,保证每次运行结果一致。

**怎么做:**
- 导入 `numpy`(数值计算库,用来做矩阵运算)和 `matplotlib.pyplot`(画图库);
- 调用 `np.random.seed(42)` 固定随机数种子——就像掷骰子前先设定好点数序列,保证每次重跑代码得到完全相同的随机数,便于复现实验结果。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Original ResNet Block(原始 ResNet 块)

```
x → Conv → BN → ReLU → Conv → BN → (+) → ReLU → output
    ↓                                  ↑
    └──────────── identity ────────────┘
```

#### 💻 代码解读

**做什么:** 用 NumPy 从零实现原版 ResNet 残差块(后激活 post-activation 结构),并用一个 8 维随机向量测试它的前向传播。

**怎么做:**
- 定义 `relu(x)` 函数:小于 0 的值全部置零,这是最常用的激活函数;
- 定义 `batch_norm_1d(x)`:简化版批归一化(Batch Normalization),把数据减去均值、除以标准差,再用 `gamma`、`beta` 缩放平移——好比把不同班级的分数都换算成"标准分"再比较;
- 定义 `OriginalResidualBlock` 类:`forward` 方法按原版顺序执行 **Conv → BN → ReLU → Conv → BN**,然后 `out + x` 加上恒等捷径(残差连接),**最后再过一次 ReLU**——这个加法之后的 ReLU 正是原版设计的"堵点";
- 创建 `original_block` 实例,输入随机向量 `x`,打印输入和输出的前 4 个数值,验证块能正常工作。


In [ ]:
def relu(x):
    # ReLU激活:逐元素取max(0,x),负值被截断为0——这正是它会"阻断"恒等路径的原因
    return np.maximum(0, x)

def batch_norm_1d(x, gamma=1.0, beta=0.0, eps=1e-5):
    """Simplified batch normalization for 1D"""
    mean = np.mean(x)
    var = np.var(x)
    # 标准化:减均值除以标准差;分母加eps防止方差为0时除零
    x_normalized = (x - mean) / np.sqrt(var + eps)
    # gamma/beta是可学习的缩放和平移参数,让网络能恢复任意分布
    return gamma * x_normalized + beta

class OriginalResidualBlock:
    """Original ResNet block (post-activation)"""
    def __init__(self, dim):
        self.dim = dim
        # Two layers
        # 用小尺度(0.01)随机初始化权重,模拟两个"卷积"层(此处简化为全连接矩阵)
        self.W1 = np.random.randn(dim, dim) * 0.01
        self.W2 = np.random.randn(dim, dim) * 0.01
        
    def forward(self, x):
        """
        Original: x → Conv → BN → ReLU → Conv → BN → (+x) → ReLU
        """
        # First conv-bn-relu
        # np.dot即矩阵乘,形状:(dim,dim)@(dim,)->(dim,),相当于一层线性变换/卷积
        out = np.dot(self.W1, x)
        out = batch_norm_1d(out)
        out = relu(out)
        
        # Second conv-bn
        out = np.dot(self.W2, out)
        out = batch_norm_1d(out)
        
        # Add identity (residual connection)
        # 残差连接:输出=F(x)+x,网络只需学习"残差"F(x)
        out = out + x
        
        # Final ReLU (post-activation)
        # 关键缺陷:相加之后又过ReLU,恒等路径被激活函数"污染",负值信号无法原样传递
        out = relu(out)
        
        return out

# Test
original_block = OriginalResidualBlock(dim=8)
x = np.random.randn(8)
output_original = original_block.forward(x)

print(f"Input: {x[:4]}...")
print(f"Original ResNet output: {output_original[:4]}...")

## Pre-activation ResNet Block(预激活 ResNet 块)

```
x → BN → ReLU → Conv → BN → ReLU → Conv → (+) → output
    ↓                                       ↑
    └──────────── identity ─────────────────┘
```

**Key difference**: Activation BEFORE convolution, clean identity path!

**关键区别**:激活(activation)在卷积(convolution)之前,恒等路径(identity path)干净无阻!

#### 💻 代码解读

**做什么:** 实现论文提出的改进版——预激活(pre-activation)残差块,并用同一个输入 `x` 测试,和上面的原版块形成对比。

**怎么做:**
- 定义 `PreActivationResidualBlock` 类,权重初始化与原版相同(`W1`、`W2` 两个小随机矩阵);
- `forward` 方法把顺序改为 **BN → ReLU → Conv → BN → ReLU → Conv**:激活和归一化被挪到卷积**之前**,所以叫"预激活";
- 最后直接 `out + x` 返回,**加法之后不再有任何 ReLU**——恒等捷径像一条没有红绿灯的高速公路,信息可以原封不动地通过;
- 打印输出结果,并强调关键区别:干净的恒等路径(加法后没有 ReLU)。


In [ ]:
class PreActivationResidualBlock:
    """Pre-activation ResNet block (improved)"""
    def __init__(self, dim):
        self.dim = dim
        self.W1 = np.random.randn(dim, dim) * 0.01
        self.W2 = np.random.randn(dim, dim) * 0.01
        
    def forward(self, x):
        """
        Pre-activation: x → BN → ReLU → Conv → BN → ReLU → Conv → (+x)
        """
        # First bn-relu-conv
        # 预激活的核心:把BN和ReLU移到卷积"之前",顺序从Conv-BN-ReLU变为BN-ReLU-Conv
        out = batch_norm_1d(x)
        out = relu(out)
        out = np.dot(self.W1, out)
        
        # Second bn-relu-conv
        out = batch_norm_1d(out)
        out = relu(out)
        out = np.dot(self.W2, out)
        
        # Add identity (NO activation after!)
        # 相加后不再有任何操作:恒等路径完全"干净",信号和梯度都能无损直通
        # 反向传播时dL/dx=dL/dout*(1+dF/dx),其中的"1"保证梯度至少原样回传
        out = out + x
        
        return out

# Test
preact_block = PreActivationResidualBlock(dim=8)
output_preact = preact_block.forward(x)

print(f"\nPre-activation ResNet output: {output_preact[:4]}...")
print(f"\nKey difference: Clean identity path (no ReLU after addition)")

## Gradient Flow Analysis(梯度流动分析)

Why pre-activation is better:

为什么预激活(pre-activation)更好:

#### 💻 代码解读

**做什么:** 模拟梯度反向传播穿过 20 层残差块的过程,对比两种结构下梯度强度的变化,画图证明预激活结构的梯度流更通畅。

**怎么做:**
- 定义 `compute_gradient_flow` 函数:先堆叠 `num_layers` 个指定类型的残差块做前向传播,再用简化方式模拟反向传播;
- 模拟规则:梯度分成"恒等路径 + 残差路径"两股。原版(`original`)因为加法后的 ReLU 会"杀死"一部分梯度,残差路径的保留系数取 0.5~1.0 的随机数;预激活(`preact`)路径更干净,系数取 0.7~1.0,每层保留得更多;
- 对两种结构各跑 20 层,用 `np.linalg.norm` 计算每层的梯度模长(`grad_mag_original`、`grad_mag_preact`);
- 用 `plt.plot` 画出两条曲线对比,并打印梯度传到输入层时的大小——预激活的梯度明显更强,说明深层网络更容易训练。


In [ ]:
def compute_gradient_flow(block_type, num_layers=10, input_dim=8):
    """
    Simulate gradient flow through stacked residual blocks
    """
    x = np.random.randn(input_dim)
    
    # Create blocks
    if block_type == 'original':
        blocks = [OriginalResidualBlock(input_dim) for _ in range(num_layers)]
    else:
        blocks = [PreActivationResidualBlock(input_dim) for _ in range(num_layers)]
    
    # Forward pass
    activations = [x]
    current = x
    for block in blocks:
        current = block.forward(current)
        activations.append(current.copy())
    
    # Simulate backward pass (simplified gradient flow)
    # 从损失端出发,初始梯度设为全1向量,模拟反向传播的起点
    grad = np.ones(input_dim)  # Gradient from loss
    gradients = [grad]
    
    for i in range(num_layers):
        # For residual blocks: gradient splits into identity + residual path
        # Pre-activation has cleaner gradient flow
        
        if block_type == 'original':
            # Post-activation: gradient affected by ReLU derivative
            # Simplified: some gradient is killed by ReLU
            # 用0.5~1.0的随机衰减系数模拟ReLU导数(负区间导数为0)对梯度的削弱
            grad_through_residual = grad * np.random.uniform(0.5, 1.0, input_dim)
            # 残差结构的梯度=恒等路径梯度+残差路径梯度,两路相加
            grad = grad + grad_through_residual  # Identity + residual
        else:
            # Pre-activation: clean identity path
            # 预激活的衰减更小(0.7~1.0),因为恒等路径上没有ReLU阻挡
            grad_through_residual = grad * np.random.uniform(0.7, 1.0, input_dim)
            grad = grad + grad_through_residual  # Better gradient flow
        
        gradients.append(grad.copy())
    
    return activations, gradients

# Compare gradient flow
_, grad_original = compute_gradient_flow('original', num_layers=20)
_, grad_preact = compute_gradient_flow('preact', num_layers=20)

# Compute gradient magnitudes
# 列表推导式:对每层梯度向量求L2范数,得到逐层的梯度大小曲线
grad_mag_original = [np.linalg.norm(g) for g in grad_original]
grad_mag_preact = [np.linalg.norm(g) for g in grad_preact]

# Plot
plt.figure(figsize=(12, 5))
plt.plot(grad_mag_original, 'o-', label='Original ResNet (post-activation)', linewidth=2)
plt.plot(grad_mag_preact, 's-', label='Pre-activation ResNet', linewidth=2)
plt.xlabel('Layer (from output to input)', fontsize=12)
plt.ylabel('Gradient Magnitude', fontsize=12)
plt.title('Gradient Flow Comparison', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Original ResNet gradient at input: {grad_mag_original[-1]:.2f}")
print(f"Pre-activation gradient at input: {grad_mag_preact[-1]:.2f}")
print(f"\nPre-activation maintains stronger gradients!")

## Different Activation Placements(不同的激活位置)

The paper analyzes various placement options:

论文分析了多种激活放置方案:

#### 💻 代码解读

**做什么:** 以对照表形式打印论文中分析过的 4 种残差块变体,比较它们的恒等路径是否被"堵住",并给出星级评分。

**怎么做:**
- 用字典列表 `architectures` 记录 4 种结构:原版(Original)、加法后再 BN、加法前 ReLU、完全预激活(Full pre-activation),每种都写明结构公式(`structure`)、恒等路径状态(`identity`)和评分(`score`);
- 用 `for` 循环逐个打印每种结构的名称、星级、运算流程和恒等路径是否畅通;
- 最后打印结论:赢家是**完全预激活**(BN → ReLU → Conv 的顺序)——它是唯一恒等路径完全干净(CLEAN)的方案,得五星。


In [ ]:
# Visualize different architectures
# 对应论文中尝试的多种残差块变体:关键在于恒等路径是否被BN/ReLU阻挡
architectures = [
    {
        'name': 'Original',
        'structure': 'x → Conv → BN → ReLU → Conv → BN → (+x) → ReLU',
        'identity': 'Blocked by ReLU',
        'score': '★★★☆☆'
    },
    {
        'name': 'BN after addition',
        'structure': 'x → Conv → BN → ReLU → Conv → BN → (+x) → BN → ReLU',
        'identity': 'Blocked by BN & ReLU',
        'score': '★★☆☆☆'
    },
    {
        'name': 'ReLU before addition',
        'structure': 'x → BN → ReLU → Conv → BN → ReLU → Conv → ReLU → (+x)',
        'identity': 'Blocked by ReLU',
        'score': '★★☆☆☆'
    },
    {
        'name': 'Full pre-activation',
        'structure': 'x → BN → ReLU → Conv → BN → ReLU → Conv → (+x)',
        'identity': 'CLEAN! ✓',
        'score': '★★★★★'
    },
]

print("\n" + "="*80)
print("RESIDUAL BLOCK ARCHITECTURES COMPARISON")
print("="*80 + "\n")

# enumerate(...,1)从1开始编号;f-string中:20s表示左对齐占20个字符宽
for i, arch in enumerate(architectures, 1):
    print(f"{i}. {arch['name']:20s} {arch['score']}")
    print(f"   Structure: {arch['structure']}")
    print(f"   Identity path: {arch['identity']}")
    print()

print("="*80)
print("WINNER: Full pre-activation (BN → ReLU → Conv)")
print("="*80)

## Deep Network Comparison(深层网络对比)

#### 💻 代码解读

**做什么:** 搭建 50 层深的残差网络,对比原版块和预激活块在深层堆叠时信号(激活值)逐层传播的差异,并画图可视化。

**怎么做:**
- 定义 `DeepResNet` 类:根据 `block_type` 参数把 50 个 `OriginalResidualBlock` 或 `PreActivationResidualBlock` 串起来,`forward` 方法记录每一层的激活值;
- 用同一个 16 维输入 `x_input` 分别喂给 `net_original` 和 `net_preact`,收集各层激活;
- 用 `np.linalg.norm` 计算每层激活的模长(`norms_original`、`norms_preact`),观察信号逐层是放大还是衰减;
- 画两幅图:左图(`ax1`)是两种网络逐层激活强度的折线对比;右图(`ax2`)用热力图 `imshow` 显示两种网络激活值的逐维差异,红蓝颜色代表差异的正负;
- 最后打印两种网络最终输出的模长数值。


In [ ]:
class DeepResNet:
    """Stack of residual blocks"""
    def __init__(self, dim, num_blocks, block_type='preact'):
        self.blocks = []
        for _ in range(num_blocks):
            if block_type == 'preact':
                self.blocks.append(PreActivationResidualBlock(dim))
            else:
                self.blocks.append(OriginalResidualBlock(dim))
    
    def forward(self, x):
        activations = [x]
        # 逐块前向传播,并保存每层激活的副本(copy防止后续被原地修改)
        for block in self.blocks:
            x = block.forward(x)
            activations.append(x.copy())
        return x, activations

# Compare deep networks
# 堆叠50个残差块,观察深层网络中两种结构的信号传播差异
depth = 50
dim = 16
x_input = np.random.randn(dim)

net_original = DeepResNet(dim, depth, 'original')
net_preact = DeepResNet(dim, depth, 'preact')

out_original, acts_original = net_original.forward(x_input)
out_preact, acts_preact = net_preact.forward(x_input)

# Compute activation statistics
# 每层激活向量的L2范数,用来衡量信号在深度方向上是衰减还是爆炸
norms_original = [np.linalg.norm(a) for a in acts_original]
norms_preact = [np.linalg.norm(a) for a in acts_preact]

# Plot activation norms
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Activation magnitudes
ax1.plot(norms_original, label='Original ResNet', linewidth=2)
ax1.plot(norms_preact, label='Pre-activation ResNet', linewidth=2)
ax1.set_xlabel('Layer', fontsize=12)
ax1.set_ylabel('Activation Magnitude', fontsize=12)
ax1.set_title(f'Activation Flow (Depth={depth})', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Activation heatmaps
# 转置后形状:(层数+1, dim) -> (dim, 层数+1),行=特征维度,列=层,便于画热力图
acts_matrix_original = np.array(acts_original).T
acts_matrix_preact = np.array(acts_preact).T

# 画两种结构激活的逐元素差值,RdBu配色:红蓝分别表示正负差异
im = ax2.imshow(acts_matrix_preact - acts_matrix_original, cmap='RdBu', aspect='auto')
ax2.set_xlabel('Layer', fontsize=12)
ax2.set_ylabel('Feature Dimension', fontsize=12)
ax2.set_title('Difference (Pre-act - Original)', fontsize=14)
plt.colorbar(im, ax=ax2)

plt.tight_layout()
plt.show()

print(f"\nOriginal ResNet final norm: {norms_original[-1]:.4f}")
print(f"Pre-activation final norm: {norms_preact[-1]:.4f}")

## Identity Mapping Analysis(恒等映射分析)

#### 💻 代码解读

**做什么:** 做一个"恒等映射考试":把残差路径的权重全部清零,检验哪种块能让输出严格等于输入——这正是论文标题 "Identity Mappings" 的核心检验。

**怎么做:**
- 定义 `test_identity_mapping` 函数:先用 `np.zeros_like` 把块的 `W1`、`W2` 全部置零(相当于残差路径什么都不学);
- 随机生成 100 个输入 `x`,分别过一遍 `forward`,用 `np.linalg.norm(y - x)` 计算输出和输入的差距(误差);
- 对原版块 `original_test` 和预激活块 `preact_test` 各测一次,打印平均误差和标准差;
- 结论:预激活块误差更小(接近 0),因为它的恒等路径没有 ReLU 阻挡,输入能原样通过——就像快递直达不用中途拆包;原版块加法后的 ReLU 会把负数截掉,输出无法完全等于输入。


In [ ]:
def test_identity_mapping(block, num_tests=100):
    """
    Test how well the block can learn identity mapping
    (When residual path learns zero, output should equal input)
    """
    # Zero out weights (residual path learns nothing)
    # 把权重置零,模拟残差分支F(x)=0的极端情况:理想的残差块此时应输出y=x
    block.W1 = np.zeros_like(block.W1)
    block.W2 = np.zeros_like(block.W2)
    
    errors = []
    for _ in range(num_tests):
        x = np.random.randn(block.dim)
        y = block.forward(x)
        # 误差=||y-x||,衡量输出偏离恒等映射的程度;原始结构因末尾ReLU截断负值而产生误差
        error = np.linalg.norm(y - x)
        errors.append(error)
    
    return np.mean(errors), np.std(errors)

# Test both block types
original_test = OriginalResidualBlock(dim=8)
preact_test = PreActivationResidualBlock(dim=8)

mean_err_original, std_err_original = test_identity_mapping(original_test)
mean_err_preact, std_err_preact = test_identity_mapping(preact_test)

print("\nIdentity Mapping Test (residual path = 0):")
print("="*60)
print(f"Original ResNet error: {mean_err_original:.6f} ± {std_err_original:.6f}")
print(f"Pre-activation error:  {mean_err_preact:.6f} ± {std_err_preact:.6f}")
print("="*60)
print(f"\nPre-activation has {'BETTER' if mean_err_preact < mean_err_original else 'WORSE'} identity mapping!")
print("(Lower error = cleaner identity path)")

## Visualize Architecture Comparison(可视化架构对比)

#### 💻 代码解读

**做什么:** 用 matplotlib 手工绘制两张结构示意图,直观展示原版块和预激活块的内部流程,一眼看出谁的恒等路径是"畅通"的。

**怎么做:**
- 定义 `draw_block` 函数:在画布上画一条蓝色竖线代表恒等路径(identity path),右侧按顺序画一串彩色方框代表残差路径的各个操作(Conv 蓝色、BN 绿色、ReLU 黄色);
- 通过 `is_preact` 参数切换两种顺序:原版是 Conv → BN → ReLU → … → ReLU*(红色框,标记加法后那个有问题的 ReLU),预激活是 BN → ReLU → Conv → …;
- 画一个圆圈 "+" 表示两条路径汇合相加,再画绿色箭头指向输出(Output);
- 加文字标注:原版图旁写红色警告 "ReLU* blocks identity!"(ReLU 堵住了恒等路径),预激活图旁写绿色 "Clean identity!"(恒等路径畅通);
- 用 `axes[0]`、`axes[1]` 左右并排画出两张图,方便对比。


In [ ]:
# Create visual comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

def draw_block(ax, title, is_preact=False):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 12)
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    
    # Identity path (left)
    # 左侧竖线代表恒等捷径(shortcut),信号从输入直达加法节点
    ax.plot([1, 1], [1, 11], 'b-', linewidth=4, label='Identity path')
    ax.arrow(1, 10.5, 0, -0.3, head_width=0.3, head_length=0.2, fc='blue', ec='blue')
    
    # Residual path (right)
    y_pos = 11
    
    # 两种结构的区别只在残差分支内操作的排列顺序;原始结构多出的ReLU*位于加法之后
    if is_preact:
        # Pre-activation: BN → ReLU → Conv → BN → ReLU → Conv
        operations = ['BN', 'ReLU', 'Conv', 'BN', 'ReLU', 'Conv']
        colors = ['lightgreen', 'lightyellow', 'lightblue', 'lightgreen', 'lightyellow', 'lightblue']
    else:
        # Original: Conv → BN → ReLU → Conv → BN
        operations = ['Conv', 'BN', 'ReLU', 'Conv', 'BN', 'ReLU*']
        colors = ['lightblue', 'lightgreen', 'lightyellow', 'lightblue', 'lightgreen', 'lightcoral']
    
    # zip把操作名与颜色配对,enumerate同时给出序号i用于计算纵向位置
    for i, (op, color) in enumerate(zip(operations, colors)):
        y = y_pos - i * 1.5
        
        # Draw box
        width = 2
        height = 1
        ax.add_patch(plt.Rectangle((6-width/2, y-height/2), width, height, 
                                   fill=True, color=color, ec='black', linewidth=2))
        ax.text(6, y, op, ha='center', va='center', fontsize=11, fontweight='bold')
        
        # Draw arrow to next
        if i < len(operations) - 1:
            ax.arrow(6, y-height/2-0.1, 0, -0.3, head_width=0.2, head_length=0.1, 
                    fc='black', ec='black', linewidth=1.5)
    
    # Addition
    # 加法节点:恒等路径与残差路径在此汇合(y = F(x) + x)
    add_y = y_pos - len(operations) * 1.5
    ax.plot([1, 6], [add_y, add_y], 'k-', linewidth=2)
    ax.scatter([3.5], [add_y], s=500, c='white', edgecolors='black', linewidths=3, zorder=5)
    ax.text(3.5, add_y, '+', ha='center', va='center', fontsize=20, fontweight='bold', zorder=6)
    
    # Output arrow
    ax.arrow(3.5, add_y-0.3, 0, -0.5, head_width=0.3, head_length=0.2, 
            fc='green', ec='green', linewidth=3)
    ax.text(3.5, add_y-1.2, 'Output', ha='center', fontsize=12, fontweight='bold')
    
    # Input
    ax.text(1, 11.5, 'Input', ha='center', fontsize=12, fontweight='bold')
    ax.text(6, 11.5, 'Input', ha='center', fontsize=12, fontweight='bold')
    
    # Annotations
    if not is_preact:
        ax.text(8.5, add_y, 'ReLU* blocks\nidentity!', fontsize=10, color='red', 
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    else:
        ax.text(8.5, add_y, 'Clean\nidentity!', fontsize=10, color='green',
               bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

draw_block(axes[0], 'Original ResNet (Post-activation)', is_preact=False)
draw_block(axes[1], 'Pre-activation ResNet (Improved)', is_preact=True)

plt.tight_layout()
plt.show()

## Key Takeaways(核心要点)

### The Identity Mapping Problem:(恒等映射问题:)

In original ResNet:
```
y = ReLU(F(x) + x)
```
The ReLU **after addition blocks** the identity path!

在原始 ResNet 中,加法之后的 ReLU **阻断了**恒等路径(identity path)!

### Pre-activation Solution:(预激活解决方案:)

```
y = F'(x) + x
```
where F'(x) = Conv(ReLU(BN(Conv(ReLU(BN(x))))))

**Clean identity path** → better gradient flow!

其中 F'(x) = Conv(ReLU(BN(Conv(ReLU(BN(x))))))

**干净的恒等路径** → 更好的梯度流动!

### Key Changes:(关键改动:)

1. **Move BN before Conv**: `x → BN → ReLU → Conv`
2. **Remove final ReLU**: No activation after addition
3. **Result**: Identity path is truly identity

1. **把 BN 移到 Conv 之前**:`x → BN → ReLU → Conv`
2. **去掉最后的 ReLU**:加法之后不再有激活
3. **结果**:恒等路径成为真正的恒等映射

### Gradient Flow:(梯度流动:)

**Original**:
```
∂L/∂x = ∂L/∂y · (∂F/∂x + I) · ∂ReLU/∂y
```
ReLU derivative kills gradients!

**原始版本**:ReLU 的导数会扼杀梯度!

**Pre-activation**:
```
∂L/∂x = ∂L/∂y · (∂F'/∂x + I)
```
Clean gradient flow through identity!

**预激活版本**:梯度沿恒等路径畅通无阻地流动!

### Benefits:(优势:)

- ✅ **Better gradient flow**: No blocking on identity path
- ✅ **Easier optimization**: Can train deeper networks (1000+ layers)
- ✅ **Better accuracy**: Small but consistent improvement
- ✅ **Regularization**: BN before Conv acts as regularizer

- ✅ **更好的梯度流动**:恒等路径上没有阻碍
- ✅ **更易优化**:可以训练更深的网络(1000+ 层)
- ✅ **更高的准确率**:提升幅度虽小但稳定一致
- ✅ **正则化(regularization)**:Conv 之前的 BN 起到正则化作用

### Comparison:(对比:)

| Architecture | Identity Path | Gradient Flow | Performance |
|--------------|---------------|---------------|-------------|
| Original ResNet | Blocked by ReLU | Good | ★★★★☆ |
| Pre-activation | **Clean** | **Better** | ★★★★★ |

| 架构 | 恒等路径 | 梯度流动 | 性能 |
|--------------|---------------|---------------|-------------|
| 原始 ResNet | 被 ReLU 阻断 | 良好 | ★★★★☆ |
| 预激活(Pre-activation) | **干净** | **更好** | ★★★★★ |

### Implementation Tips:(实现技巧:)

1. Use pre-activation for very deep networks (>50 layers)
2. Keep original ResNet for shallower networks (backward compatibility)
3. First layer can keep post-activation (no identity yet)
4. Last layer needs post-activation for final output

1. 非常深的网络(>50 层)使用预激活(pre-activation)
2. 较浅的网络保留原始 ResNet(向后兼容)
3. 第一层可以保留后激活(post-activation)(此时还没有恒等路径)
4. 最后一层需要后激活以产生最终输出

### Results:(结果:)

- CIFAR-10: 1001-layer network trained successfully!
- ImageNet: Consistent improvements over original ResNet
- Enabled training of 1000+ layer networks

- CIFAR-10:成功训练了 1001 层的网络!
- ImageNet:相对原始 ResNet 取得一致的提升
- 使训练 1000+ 层的网络成为可能

### Why It Matters:(为什么重要:)

This paper showed that **architecture details matter**. Small changes (moving BN/ReLU) can have significant impact on trainability and performance. It's a key example of iterative improvement in deep learning research.

这篇论文表明**架构细节至关重要**。小小的改动(移动 BN/ReLU 的位置)就能对可训练性和性能产生显著影响。它是深度学习研究中迭代式改进的一个典型范例。